## Study takeaway (read first)

**Topic:** BERT masked language modeling — no training (Lecture 3)

**Walkthrough:** Load BERT -> ask_BERT([MASK] sentences) -> top predictions EN + DE (multilingual BERT) -> cloze test.

**Remember for exam:** MLM masks 15% of tokens; bidirectional context; contextual embeddings. Fine-tune: [CLS] + classification head.

**Time:** ~30min | **Exam priority:** Low | Slides: BertLab.pdf

# AdvNLP BERTlab

## Session goal
In this session we'll be using pre-trained BERT to predict missing words in English and German and to tackle a cloze test.

We're only using pre-trained BERT, so you don't need a GPU to run this notebook.
To avoid platform-specific problems, we recommend you run this notebook on Colab.

In [1]:
!pip install pytorch-pretrained-bert
import torch
from pytorch_pretrained_bert import BertTokenizer, BertModel, BertForMaskedLM


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Choose a model below. Be sure to use a **multilingual** model if you wish to use German or anything else other than English.

In [2]:
import numpy as np

#model = 'bert-base-multilingual-uncased'
#model = 'bert-base-multilingual-cased'
#model = 'bert-base-cased'
model = 'bert-large-uncased'
do_lower_case = True

tokenizer = BertTokenizer.from_pretrained(model, do_lower_case=do_lower_case)
language_model = BertForMaskedLM.from_pretrained(model)
language_model.eval()

100%|██████████| 1248501532/1248501532 [01:31<00:00, 13652443.03B/s]


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1, inplace=False

Enter your text below. Populate the **candidates** list if you want BERT to choose out of a set of predefined words.

In [3]:
def ask_BERT (text, candidates, language_model, tokenizer):
    tokenized_text = tokenizer.tokenize(text)

    masked_index = []


    for i, token in enumerate(tokenized_text):
      if token == '_':
        masked_index.append(i)
        tokenized_text[i]= '[MASK]'


    candidates_ids = tokenizer.convert_tokens_to_ids(candidates)

    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)

    segments_ids = [0] * len(tokenized_text)

    tokens_tensor = torch.tensor([indexed_tokens])
    segments_tensors = torch.tensor([segments_ids])





    predictions = language_model(tokens_tensor, segments_tensors)


    if len(candidates)>0:
      predictions_candidates = predictions[0, masked_index, candidates_ids]
      probs = (predictions_candidates.softmax(dim=0))

      probabilities = (probs.detach().numpy())
      tokens = tokenizer.convert_ids_to_tokens(candidates_ids)

      tuple_list = ([(x, probabilities[i]) for i, x in enumerate(tokens)])

      sorted_tuple_list = sorted(tuple_list, key=lambda x: x[1], reverse=True)
      print (text)
      print (sorted_tuple_list)

      answer_idx = torch.argmax(predictions_candidates).item()
      print(f'The most likely word is "{candidates[answer_idx]}".')

    else:
      predictions_candidates = predictions[0, masked_index, range(predictions.shape[-1])]

      probs = (predictions_candidates.softmax(dim=0))
      max_prob=probs.max(dim=0)
      threshold = 0.01
      indices = ((np.where(probs>threshold)))

      probabilities = ((probs[probs>threshold]).detach().numpy())
      tokens = tokenizer.convert_ids_to_tokens(indices[0])

      tuple_list = ([(x, probabilities[i]) for i, x in enumerate(tokens)])

      sorted_tuple_list = sorted(tuple_list, key=lambda x: x[1], reverse=True)
      print (text)
      print (sorted_tuple_list)



In [4]:
text = '”I am the first to arrive.” She thought and came to her desk.'
text = text+' She was surprised to find a bunch of flowers on it.'
text = text+' They were fresh. She _ them and they were sweet. She looked around for a vase to put them in.'
candidates = ['smelled', 'ate', 'took', 'held']

ask_BERT (text, candidates, language_model, tokenizer)

”I am the first to arrive.” She thought and came to her desk. She was surprised to find a bunch of flowers on it. They were fresh. She _ them and they were sweet. She looked around for a vase to put them in.
[('smelled', np.float32(0.6510978)), ('ate', np.float32(0.15703417)), ('took', np.float32(0.1222518)), ('held', np.float32(0.069616266))]
The most likely word is "smelled".


In [5]:
text = 'Nancy had just got a new job in a company.\
        Monday was the first day she went to work, so she \
        was very _ and arrived early.'
candidates = ['depressed', 'encouraged', 'excited', 'surprised']
ask_BERT (text, candidates, language_model, tokenizer)

Nancy had just got a new job in a company.        Monday was the first day she went to work, so she         was very _ and arrived early.
[('excited', np.float32(0.85088223)), ('surprised', np.float32(0.13961668)), ('encouraged', np.float32(0.0048667053)), ('depressed', np.float32(0.004634446))]
The most likely word is "excited".


In [6]:
text = 'Nancy had just got a new job in a company.\
        Monday was the first day she went to work, so she \
        was very excited and arrived early.'
text = text+'I am the _ to arrive.” She thought and came to her desk.'
candidates = ['last', 'second', 'third', 'first']
ask_BERT (text, candidates, language_model, tokenizer)

Nancy had just got a new job in a company.        Monday was the first day she went to work, so she         was very excited and arrived early.I am the _ to arrive.” She thought and came to her desk.
[('first', np.float32(0.7606356)), ('last', np.float32(0.14107886)), ('second', np.float32(0.09030674)), ('third', np.float32(0.007978823))]
The most likely word is "first".


For German, we need Multilingual BERT.

In [7]:
import numpy as np

model = 'bert-base-multilingual-cased'

tokenizer = BertTokenizer.from_pretrained(model, do_lower_case=do_lower_case)
language_model = BertForMaskedLM.from_pretrained(model)
language_model.eval()

100%|██████████| 662804195/662804195 [00:56<00:00, 11829973.36B/s]


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1, inplace=False)
        

In [8]:
text = 'Wir gehen durch _ Wald.'
ask_BERT (text, [], language_model, tokenizer)

Wir gehen durch _ Wald.
[('den', np.float32(0.8579788)), ('einen', np.float32(0.054590646)), ('ein', np.float32(0.016729726)), ('diesen', np.float32(0.015646763))]


In [9]:
text = 'Die Trophäe würde nicht in den braunen Koffer passen, weil _ zu groß war.'
ask_BERT (text, [], language_model, tokenizer)

Die Trophäe würde nicht in den braunen Koffer passen, weil _ zu groß war.
[('er', np.float32(0.39350224)), ('es', np.float32(0.17258963)), ('sie', np.float32(0.16843055)), ('diese', np.float32(0.029279953)), ('man', np.float32(0.019921355)), ('dieser', np.float32(0.019807879))]


Below you can find all the text used in the slides. Pass each string to *ask_BERT* to see whether Multilingual BERT knows enough about German grammar. Be sure to switch back to the monolingual model if you wish to use the English language examples below.

In [10]:
# German examples
text = 'Bitte legen Sie es auf _ Schreibtisch.'
text = 'Es ist auf _ Schreibtisch.'
text = 'Wir gehen durch _ Wald.'
text = 'Sie kommen aus _ Schweiz.'
text = 'Ich ging trotz _ Erkältung zur Arbeit.'

text = 'Die Trophäe würde nicht in den braunen Koffer passen, weil _ zu groß war.'
text = 'Le trophée ne rentrait pas dans la valise marron parce qu\'_ était trop grande.'
text = 'The trophy would not fit in the brown suitcase because it was too big. What was too big, the trophy or the suitcase? The _.'

text = 'Mark visited Janet\'s grave in 1765. At that time, _ had been traveling for five years.'

text = 'Mark visited Janet\'s grave in 1765. Mark was alive, Janet was dead. At that time, _ had been traveling for five years.'

text = 'Nancy had just got a new job in a company.\
        Monday was the first day she went to work, so she \
        was very _ and arrived early.'
candidates = ['depressed', 'encouraged', 'excited', 'surprised']

text = 'She _ the door open and found nobody there.'
candidates = ['turned', 'pushed', 'knocked', 'forced']


text = 'Ich ging trotz _ Erkältung zur Arbeit.'


text='Ich träume von _ sprechenden Delphin.'

text = 'Ich träume _, mit einem Delfin zu sprechen.'

text = 'Die Trophäe würde nicht in den braunen Koffer passen, weil _ zu klein ist.'



text = 'Mark visited Janet\'s grave in 1765. At that time, _ had been dead for five years.'

text = 'Janet had died. Mark visited Janet\'s grave in 1765 (Janet had died). At that time, _ had been traveling for five years.'


text = 'Die Trophäe würde nicht in den braunen Koffer passen, weil _ zu klein ist.'


# English examples
text = 'She _ the door open and found nobody there.'
candidates = []
candidates = ['turned', 'pushed', 'knocked', 'forced']

text = 'Mark visited Janet\'s grave in 1765. Mark was alive, Janet was dead. At that time, _ had been traveling for five years.'
candidates = []

text = 'Nancy had just got a new job in a company.\
        Monday was the first day she went to work, so she \
        was very excited and arrived early.'
candidates = ['depressed', 'encouraged', 'excited', 'surprised']
text = ' She pushed the door open and found nobody there.'
candidates = ['turned', 'pushed', 'knocked', 'forced']
text = '”I am the _ to arrive.” She thought and came to her desk.'
candidates = ['last', 'second', 'third', 'first']
text = text+' She was surprised to find a bunch of flowers on it.'
text = text+' They were fresh. She _ them and they were sweet. She looked around for a vase to put them in.'
candidates = ['smelled', 'ate', 'took', 'held']

In [12]:
text = 'Bitte legen Sie es auf _ Schreibtisch.'
ask_BERT (text, [], language_model, tokenizer)

Bitte legen Sie es auf _ Schreibtisch.
[('den', np.float32(0.55494684)), ('dem', np.float32(0.18432148)), ('einen', np.float32(0.09111887)), ('einem', np.float32(0.08714555)), ('der', np.float32(0.015587591))]


In [13]:
text = 'Die grössten Einbussen bei den Grossunternehmen gab es neben dem Personalvermittler Adecco bei _ Finanzwerten Partners Group, UBS und Julius Bär, die zeitweise zwischen 8 und 9 Prozent einbrachen.'
ask_BERT (text, [], language_model, tokenizer)

Die grössten Einbussen bei den Grossunternehmen gab es neben dem Personalvermittler Adecco bei _ Finanzwerten Partners Group, UBS und Julius Bär, die zeitweise zwischen 8 und 9 Prozent einbrachen.
[('den', np.float32(0.98490936))]


In [14]:
text = 'Die grössten Einbussen bei den Grossunternehmen gab es neben _ Personalvermittler Adecco bei den Finanzwerten Partners Group, UBS und Julius Bär, die zeitweise zwischen 8 und 9 Prozent einbrachen.'
ask_BERT (text, [], language_model, tokenizer)

Die grössten Einbussen bei den Grossunternehmen gab es neben _ Personalvermittler Adecco bei den Finanzwerten Partners Group, UBS und Julius Bär, die zeitweise zwischen 8 und 9 Prozent einbrachen.
[('dem', np.float32(0.9273633)), ('der', np.float32(0.030344239)), ('den', np.float32(0.020837616))]
